# Distributions and outliers

Target: `OD_PAX`, passengers on one origin-destination route in one departure month.
The forecasting file is the modeling table. Plots use a 50,000-row sample (`random_state=9`). Summary numbers use the full file.

Model family this section supports: **gradient-boosted trees**. The passenger distribution is skewed, and a route-month table is not one time series.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
pax = pd.read_csv(root / "data" / "raw" / "AA_PAX_FORECASTING.csv")
print(f"{len(pax):,} rows, {pax.shape[1]} columns")
print("Origins:", pax["ORG_AIRPRT"].nunique(), "Destinations:", pax["DES_AIRPRT"].nunique())


## The target

`OD_PAX` is heavily right-skewed. A few very busy route-months pull the mean far above the median, so a squared-error linear model spends its fit on that tail. Tree splits are not thrown off by the scale.

In [ ]:
y = pax["OD_PAX"]
print(
    f"min={y.min():.2f}  median={y.median():,.1f}  mean={y.mean():,.1f}  "
    f"max={y.max():,.1f}  skew={y.skew():.2f}"
)
print("Share of rows above 10x the median:", f"{(y > 10 * y.median()).mean():.1%}")


On the full file the minimum is 0.13, the median is about 503, the mean is about 2,765, the maximum is about 140,194, and the skew is 5.32. Values below 1 mean these are estimated counts, not raw boardings. The mean is more than five times the median.

In [ ]:
sample = pax.sample(n=50_000, random_state=9)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(np.log10(sample["OD_PAX"]), bins=40, color="#1f4e79")
axes[0].set_title("log10(OD_PAX), 50k sample")
axes[0].set_xlabel("log10(passengers)")
month_median = pax.groupby("DEP_MONTH")["OD_PAX"].median()
axes[1].bar(month_median.index.astype(str), month_median.to_numpy(), color="#1f4e79")
axes[1].set_title("Median OD_PAX by departure month, full file")
axes[1].set_xlabel("DEP_MONTH")
plt.tight_layout()
plt.show()


## Month, year, and linearity

Median passengers rise from about 400 in February to about 616 in July, then ease off. That is a seasonal bump, not a straight line a linear model should be trusted to draw through every route.

At row level, `DEP_MONTH` correlates with `OD_PAX` at about 0.01, and `DEP_YEAR` at about 0.05. The calendar effect is real and small next to which route the row is. Route identity dominates the distribution.

In [ ]:
print("Row-level corr with OD_PAX:")
print("  DEP_MONTH", round(pax["DEP_MONTH"].corr(pax["OD_PAX"]), 3))
print("  DEP_YEAR ", round(pax["DEP_YEAR"].corr(pax["OD_PAX"]), 3))
print("Rows by year:")
print(pax.groupby("DEP_YEAR").size().to_string())
print("2026 months present:", sorted(pax.loc[pax["DEP_YEAR"].eq(2026), "DEP_MONTH"].unique().tolist()))


## This is not one time series

There are 150 origins and 150 destinations. A full year has about 205,000 rows. 2020 is lower (about 186,000). 2026 stops at July (about 120,000 rows). Each row is a route in a month, so the file is a panel.

`DEP_MONTH` and `DEP_YEAR` are a real time axis for a route, and they are not enough for ARIMA. ARIMA needs one ordered series. Forecasting every route-month is a tabular problem: categories for the airports, a seasonal month, and a year. That is the setting gradient-boosted trees are built for.

**Gradient-boosted trees**, because the target is skewed (skew 5.32) and the structure is a large categorical panel rather than one line through time.